In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

def time_to_minutes(t):
    if pd.isnull(t): return np.nan
    h, m = map(int, str(t).split(':'))
    return h * 60 + m

# ==========================================
# ADVANCED LEAKAGE-FREE TARGET ENCODING
# ==========================================
def get_oof_encoding(train_df, test_df, col, target='demand', folds=5):
    """
    Calculates the historical mean of a category using K-Fold Out-of-Fold (OOF).
    This completely prevents target leakage during validation and training.
    """
    train_encoded = pd.Series(np.nan, index=train_df.index)
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(train_df):
        # Calculate mean strictly on the train folds
        means = train_df.iloc[train_idx].groupby(col)[target].mean()
        # Map to the validation fold
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][col].map(means)

    # For test data, we safely use the entire training set's historical mean
    global_means = train_df.groupby(col)[target].mean()
    test_encoded = test_df[col].map(global_means)

    # Fill any missing/unseen categories with the global average demand
    global_target_mean = train_df[target].mean()
    train_encoded = train_encoded.fillna(global_target_mean)
    test_encoded = test_encoded.fillna(global_target_mean)

    return train_encoded, test_encoded


def create_v3_features(train_path, test_path):
    print("1. Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df['is_test'] = 0
    test_df['is_test'] = 1

    combined = pd.concat([train_df, test_df], ignore_index=True)

    # ==========================================
    # UPGRADE 1: Massive Time Feature Expansion
    # ==========================================
    print("2. Engineering Rich Temporal Features...")
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else np.nan)
    combined['minutes'] = combined['timestamp'].apply(time_to_minutes)

    combined['hour_sin'] = np.sin(2 * np.pi * combined['hour'] / 24)
    combined['hour_cos'] = np.cos(2 * np.pi * combined['hour'] / 24)
    combined['minute_sin'] = np.sin(2 * np.pi * combined['minutes'] / 1440)
    combined['minute_cos'] = np.cos(2 * np.pi * combined['minutes'] / 1440)

    # Traffic behavior flags
    combined['is_peak_morning'] = combined['hour'].between(7, 10).astype(int)
    combined['is_peak_evening'] = combined['hour'].between(16, 20).astype(int)
    combined['is_night'] = combined['hour'].between(0, 5).astype(int)

    # Time Bins
    combined['time_bin'] = pd.cut(
        combined['hour'],
        bins=[-1, 6, 10, 14, 18, 22, 25], # -1 and 25 to catch edge cases
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)

    # ==========================================
    # UPGRADE 2: Geohash Hierarchies
    # ==========================================
    print("3. Extracting Geohash Spatial Hierarchies (City -> District -> Block)...")
    # A geohash of length 6 is roughly 1.2km x 0.6km. Length 4 is ~39km x 19km.
    combined['geo_4'] = combined['geohash'].astype(str).str[:4]
    combined['geo_5'] = combined['geohash'].astype(str).str[:5]
    combined['geo_6'] = combined['geohash'].astype(str).str[:6]

    # ==========================================
    # UPGRADE 3: Interaction Features
    # ==========================================
    print("4. Engineering Domain Interactions...")
    combined['LargeVehicles_num'] = combined['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(-1)
    combined['Landmarks'] = combined['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(-1)

    # Missing value handling (Better than pure median imputation)
    combined['temp_is_missing'] = combined['Temperature'].isna().astype(int)
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].median())

    # Cross-Features
    combined['road_capacity'] = combined['NumberofLanes'] * (combined['LargeVehicles_num'] + 1)
    combined['temp_hour'] = combined['Temperature'] * combined['hour']
    combined['road_weather'] = combined['RoadType'].astype(str) + "_" + combined['Weather'].astype(str)

    # ==========================================
    # Split for Target Encoding & Label Encoding
    # ==========================================
    train_full = combined[combined['is_test'] == 0].drop(columns=['is_test', 'Index', 'timestamp', 'minutes'])
    test_full = combined[combined['is_test'] == 1].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'demand'])

    # UPGRADE 4: Apply OOF Target Encoding
    print("5. Applying OOF Target Encodings...")
    encode_cols = ['geohash', 'geo_5', 'hour', 'time_bin']
    for col in encode_cols:
        tr_enc, te_enc = get_oof_encoding(train_full, test_full, col, target='demand')
        train_full[f'{col}_mean_enc'] = tr_enc
        test_full[f'{col}_mean_enc'] = te_enc

    # Format Categoricals for Models
    cat_features = ['geohash', 'geo_4', 'geo_5', 'geo_6', 'RoadType', 'Weather', 'road_weather', 'LargeVehicles']
    for col in cat_features:
        train_full[col] = train_full[col].fillna('Missing').astype(str)
        test_full[col] = test_full[col].fillna('Missing').astype(str)

    lgb_train = train_full.copy()
    lgb_test = test_full.copy()
    for col in cat_features:
        le = LabelEncoder()
        # Fit on both to handle unseen categories
        le.fit(pd.concat([train_full[col], test_full[col]]))
        lgb_train[col] = le.transform(lgb_train[col])
        lgb_test[col] = le.transform(lgb_test[col])

    return train_full, test_full, lgb_train, lgb_test, cat_features


if __name__ == "__main__":
    cb_train, cb_test, lgb_train, lgb_test, cat_cols = create_v3_features('dataset/train.csv', 'dataset/test.csv')

    # UPGRADE 5: Safe Validation Split
    # ==========================================
    max_day = cb_train['day'].max()

    y_full = cb_train['demand']
    # Validate strictly on the final available day, train on everything prior
    val_mask = cb_train['day'] == max_day
    train_mask = cb_train['day'] < max_day

    y_tr, y_va = y_full[train_mask], y_full[val_mask]

    X_cb_tr = cb_train[train_mask].drop(columns=['day', 'demand'])
    X_cb_va = cb_train[val_mask].drop(columns=['day', 'demand'])
    X_cb_full = cb_train.drop(columns=['day', 'demand'])
    X_cb_test = cb_test.drop(columns=['day'])

    X_lgb_tr = lgb_train[train_mask].drop(columns=['day', 'demand'])
    X_lgb_va = lgb_train[val_mask].drop(columns=['day', 'demand'])
    X_lgb_full = lgb_train.drop(columns=['day', 'demand'])
    X_lgb_test = lgb_test.drop(columns=['day'])

    # ---------------------------------------------------------
    # MODEL 1: TUNED CATBOOST (The Heavyweight)
    # ---------------------------------------------------------
    print("\n--- Training CatBoost Model ---")
    # UPGRADE 6: Domain-specific CatBoost tuning
    cb_model = CatBoostRegressor(
        iterations=12000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        random_strength=1,
        boosting_type='Ordered', # CRITICAL: Prevents target leakage natively
        task_type='GPU',
        eval_metric='RMSE',
        random_seed=42
    )
    cb_model.fit(X_cb_tr, y_tr, eval_set=(X_cb_va, y_va), cat_features=cat_cols, early_stopping_rounds=4000, verbose=500)

    print("Retraining CatBoost on 100% data...")
    cb_final = CatBoostRegressor(
        iterations=cb_model.get_best_iteration(), learning_rate=0.03, depth=8,
        l2_leaf_reg=5, random_strength=1, boosting_type='Ordered', task_type='GPU', random_seed=42
    )
    cb_final.fit(X_cb_full, y_full, cat_features=cat_cols, verbose=False)
    cb_preds = cb_final.predict(X_cb_test)

    # ---------------------------------------------------------
    # MODEL 2: LIGHTGBM (The Agile Support)
    # ---------------------------------------------------------
    print("\n--- Training LightGBM Model ---")
    d_train = lgb.Dataset(X_lgb_tr, label=y_tr)
    d_val = lgb.Dataset(X_lgb_va, label=y_va, reference=d_train)

    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.03,
        'max_depth': 8,
        'num_leaves': 63,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'device_type': 'gpu', # Change to 'cpu' if necessary
        'verbose': -1,
        'seed': 42
    }

    lgb_model = lgb.train(
        lgb_params, d_train, num_boost_round=12000, valid_sets=[d_val],
        callbacks=[lgb.early_stopping(stopping_rounds=4000), lgb.log_evaluation(500)]
    )

    print("Retraining LightGBM on 100% data...")
    d_train_full = lgb.Dataset(X_lgb_full, label=y_full)
    lgb_final = lgb.train(lgb_params, d_train_full, num_boost_round=lgb_model.best_iteration)
    lgb_preds = lgb_final.predict(X_lgb_test)

    # ==========================================
    # UPGRADE 7: Weighted Ensemble & Bulletproof Save
    # ==========================================
    print("\n--- Finalizing Predictions ---")

    # 1. Clip predictions to prevent negative demand
    cb_preds = np.clip(cb_preds, 0, None)
    lgb_preds = np.clip(lgb_preds, 0, None)

    # 2. The 70/30 Split
    final_predictions = (cb_preds * 0.70) + (lgb_preds * 0.30)

    # 3. Load the full test set to get the complete Index
    test_raw = pd.read_csv('dataset/test.csv')

    # 4. Map predictions directly to all 41,778 rows
    final_sub = pd.DataFrame({
        'Index': test_raw['Index'],
        'demand': final_predictions
    })

    # 5. Fill any accidental NaNs with the mean
    safe_mean = final_sub['demand'].mean()
    final_sub['demand'] = final_sub['demand'].fillna(safe_mean)

    # 6. Save with strict formatting
    final_sub.to_csv('submission_v3_optuna12.csv', index=False, float_format='%.6f')

    print("\n" + "="*40)
    print(f"SUCCESS! Saved 'submission_v3_optuna12.csv'.")
    print(f"Final row count: {len(final_sub)} (Matches test.csv exactly)")
    print("="*40 + "\n")

1. Loading datasets...
2. Engineering Rich Temporal Features...
3. Extracting Geohash Spatial Hierarchies (City -> District -> Block)...
4. Engineering Domain Interactions...
5. Applying OOF Target Encodings...

--- Training CatBoost Model ---
0:	learn: 0.1380823	test: 0.1417789	best: 0.1417789 (0)	total: 87.3ms	remaining: 17m 26s
500:	learn: 0.0342571	test: 0.0678259	best: 0.0646795 (119)	total: 39.3s	remaining: 15m 2s
1000:	learn: 0.0326787	test: 0.0692392	best: 0.0646795 (119)	total: 1m 13s	remaining: 13m 24s
1500:	learn: 0.0319082	test: 0.0697167	best: 0.0646795 (119)	total: 1m 49s	remaining: 12m 44s
2000:	learn: 0.0312626	test: 0.0702437	best: 0.0646795 (119)	total: 2m 26s	remaining: 12m 13s
2500:	learn: 0.0305387	test: 0.0706921	best: 0.0646795 (119)	total: 3m 6s	remaining: 11m 47s
3000:	learn: 0.0300513	test: 0.0710934	best: 0.0646795 (119)	total: 3m 45s	remaining: 11m 17s
3500:	learn: 0.0296476	test: 0.0714254	best: 0.0646795 (119)	total: 4m 25s	remaining: 10m 44s
4000:	learn: 

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

def time_to_minutes(t):
    if pd.isnull(t): return np.nan
    h, m = map(int, str(t).split(':'))
    return h * 60 + m

# ==========================================
# 1. LEAKAGE-FREE TARGET ENCODING
# ==========================================
def get_oof_encoding(train_df, test_df, col, target='demand', folds=5):
    train_encoded = pd.Series(np.nan, index=train_df.index)
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(train_df):
        means = train_df.iloc[train_idx].groupby(col)[target].mean()
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][col].map(means)

    global_means = train_df.groupby(col)[target].mean()
    test_encoded = test_df[col].map(global_means)

    global_target_mean = train_df[target].mean()
    train_encoded = train_encoded.fillna(global_target_mean)
    test_encoded = test_encoded.fillna(global_target_mean)

    return train_encoded, test_encoded

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
def create_v3_features(train_path, test_path):
    print("1. Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df['is_test'] = 0
    test_df['is_test'] = 1

    combined = pd.concat([train_df, test_df], ignore_index=True)

    print("2. Engineering Rich Temporal Features...")
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else np.nan)
    combined['minutes'] = combined['timestamp'].apply(time_to_minutes)

    combined['hour_sin'] = np.sin(2 * np.pi * combined['hour'] / 24)
    combined['hour_cos'] = np.cos(2 * np.pi * combined['hour'] / 24)

    combined['is_peak_morning'] = combined['hour'].between(7, 10).astype(int)
    combined['is_peak_evening'] = combined['hour'].between(16, 20).astype(int)
    combined['is_night'] = combined['hour'].between(0, 5).astype(int)

    combined['time_bin'] = pd.cut(
        combined['hour'], bins=[-1, 6, 10, 14, 18, 22, 25], labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)

    print("3. Extracting Geohash Spatial Hierarchies...")
    combined['geo_4'] = combined['geohash'].astype(str).str[:4]
    combined['geo_5'] = combined['geohash'].astype(str).str[:5]
    combined['geo_6'] = combined['geohash'].astype(str).str[:6]

    print("4. Engineering Domain Interactions...")
    combined['LargeVehicles_num'] = combined['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(-1)
    combined['Landmarks'] = combined['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(-1)

    combined['temp_is_missing'] = combined['Temperature'].isna().astype(int)
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].median())

    combined['road_capacity'] = combined['NumberofLanes'] * (combined['LargeVehicles_num'] + 1)
    combined['temp_hour'] = combined['Temperature'] * combined['hour']
    combined['road_weather'] = combined['RoadType'].astype(str) + "_" + combined['Weather'].astype(str)

    train_full = combined[combined['is_test'] == 0].drop(columns=['is_test', 'Index', 'timestamp', 'minutes'])
    test_full = combined[combined['is_test'] == 1].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'demand'])

    print("5. Applying OOF Target Encodings...")
    encode_cols = ['geohash', 'geo_5', 'hour', 'time_bin']
    for col in encode_cols:
        tr_enc, te_enc = get_oof_encoding(train_full, test_full, col, target='demand')
        train_full[f'{col}_mean_enc'] = tr_enc
        test_full[f'{col}_mean_enc'] = te_enc

    print("6. Formatting Categoricals (OrdinalEncoder Fix)...")
    cat_features = ['geohash', 'geo_4', 'geo_5', 'geo_6', 'RoadType', 'Weather', 'road_weather', 'LargeVehicles']
    for col in cat_features:
        train_full[col] = train_full[col].fillna('Missing').astype(str)
        test_full[col] = test_full[col].fillna('Missing').astype(str)

    model_train = train_full.copy()
    model_test = test_full.copy()

    # FIXED: OrdinalEncoder prevents the "Unknown label type: continuous" error
    for col in cat_features:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        unique_vals = pd.concat([train_full[col], test_full[col]]).unique().reshape(-1, 1)
        oe.fit(unique_vals)

        # Cast to int so LightGBM recognizes them correctly
        model_train[col] = oe.transform(train_full[[col]]).astype(int)
        model_test[col] = oe.transform(test_full[[col]]).astype(int)

    return model_train, model_test, cat_features

# ==========================================
# 3. OPTUNA TUNING FUNCTIONS
# ==========================================
def tune_lightgbm(X_train, y_train, X_val, y_val, cat_cols):
    print("\n--- Tuning LightGBM with Optuna ---")
    d_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
    d_val = lgb.Dataset(X_val, label=y_val, reference=d_train, categorical_feature=cat_cols)

    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'verbosity': -1,
            'device_type': 'cpu', # Change to 'cpu' if no GPU available
            'feature_pre_filter': False, # FIXED: Prevents LightGBM crash
            'seed': 42,

            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 10),
            'num_leaves': trial.suggest_int('num_leaves', 31, 127),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.9),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        }

        model = lgb.train(
            params, d_train, valid_sets=[d_val], num_boost_round=20000,
            callbacks=[lgb.early_stopping(stopping_rounds=2000, verbose=False)]
        )
        preds = model.predict(X_val)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=300)
    print(f"Best LightGBM Params: {study.best_params}")
    return study.best_params

def tune_catboost(X_train, y_train, X_val, y_val, cat_cols):
    print("\n--- Tuning CatBoost with Optuna ---")
    def objective(trial):
        params = {
            'iterations': 10000,
            'task_type': 'GPU',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'boosting_type': 'Plain', # Faster for tuning

            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.06, log=True),
            'depth': trial.suggest_int('depth', 5, 9),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'random_strength': trial.suggest_float('random_strength', 0.1, 2.0),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0)
        }

        model = CatBoostRegressor(**params)
        model.fit(
            X_train, y_train, eval_set=(X_val, y_val),
            cat_features=cat_cols, early_stopping_rounds=2000, verbose=False
        )
        preds = model.predict(X_val)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=250)
    print(f"Best CatBoost Params: {study.best_params}")
    return study.best_params


# ==========================================
# 4. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    df_train, df_test, cat_cols = create_v3_features('dataset/train.csv', 'dataset/test.csv')

    # FIXED: Reverted to safe 1-day split to prevent "Labels variable is empty"
    max_day = df_train['day'].max()
    y_full = df_train['demand']
    val_mask = df_train['day'] == max_day
    train_mask = df_train['day'] < max_day

    y_tr, y_va = y_full[train_mask], y_full[val_mask]

    X_tr = df_train[train_mask].drop(columns=['day', 'demand'])
    X_va = df_train[val_mask].drop(columns=['day', 'demand'])
    X_full = df_train.drop(columns=['day', 'demand'])
    X_te = df_test.drop(columns=['day'])

    # ---------------------------------------------------------
    # RUN OPTUNA (Comment these two lines out after you find your best params!)
    # ---------------------------------------------------------
    best_lgb_params = tune_lightgbm(X_tr, y_tr, X_va, y_va, cat_cols)
    best_cb_params = tune_catboost(X_tr, y_tr, X_va, y_va, cat_cols)

    # ---------------------------------------------------------
    # FINAL MODELS
    # ---------------------------------------------------------
    print("\n--- Training Final CatBoost Model ---")
    final_cb_params = best_cb_params.copy()
    final_cb_params.update({
        'iterations': 20000,
        'boosting_type': 'Ordered', # Safe target encoding for the final run
        'task_type': 'GPU',
        'eval_metric': 'RMSE',
        'random_seed': 42
    })

    cb_model = CatBoostRegressor(**final_cb_params)
    cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_cols, early_stopping_rounds=2000, verbose=250)

    print("Retraining CatBoost on 100% data...")
    final_cb_params['iterations'] = cb_model.get_best_iteration()
    cb_final = CatBoostRegressor(**final_cb_params)
    cb_final.fit(X_full, y_full, cat_features=cat_cols, verbose=False)
    cb_preds = cb_final.predict(X_te)


    print("\n--- Training Final LightGBM Model ---")
    d_train = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    d_val = lgb.Dataset(X_va, label=y_va, reference=d_train, categorical_feature=cat_cols)

    final_lgb_params = best_lgb_params.copy()
    final_lgb_params.update({
        'objective': 'regression',
        'metric': 'rmse',
        'device_type': 'cpu',
        'feature_pre_filter': False,
        'verbosity': -1,
        'seed': 42
    })

    lgb_model = lgb.train(
        final_lgb_params, d_train, num_boost_round=10000, valid_sets=[d_val],
        callbacks=[lgb.early_stopping(stopping_rounds=2000), lgb.log_evaluation(250)]
    )

    print("Retraining LightGBM on 100% data...")
    d_train_full = lgb.Dataset(X_full, label=y_full, categorical_feature=cat_cols)
    lgb_final = lgb.train(final_lgb_params, d_train_full, num_boost_round=lgb_model.best_iteration)
    lgb_preds = lgb_final.predict(X_te)

    # ---------------------------------------------------------
    # FINAL ENSEMBLE (FIXED FOR ALL 41,778 ROWS)
    # ---------------------------------------------------------
    print("\n--- Finalizing Predictions ---")
    cb_preds = np.clip(cb_preds, 0, None)
    lgb_preds = np.clip(lgb_preds, 0, None)

    # The 70/30 Split
    final_predictions = (cb_preds * 0.70) + (lgb_preds * 0.30)

    test_raw = pd.read_csv('dataset/test.csv')

    final_sub = pd.DataFrame({
        'Index': test_raw['Index'],
        'demand': final_predictions
    })

    safe_mean = final_sub['demand'].mean()
    final_sub['demand'] = final_sub['demand'].fillna(safe_mean)

    final_sub.to_csv('submission_v3_optuna13.csv', index=False, float_format='%.6f')

    print("\n" + "="*40)
    print(f"SUCCESS! Saved 'submission_v3_optuna13.csv'.")
    print(f"Final row count: {len(final_sub)} (Matches full test.csv exactly)")
    print("="*40 + "\n")

1. Loading datasets...
2. Engineering Rich Temporal Features...
3. Extracting Geohash Spatial Hierarchies...
4. Engineering Domain Interactions...
5. Applying OOF Target Encodings...
6. Formatting Categoricals (OrdinalEncoder Fix)...


[I 2026-06-04 19:41:28,367] A new study created in memory with name: no-name-ac230db2-e6d7-4542-9a57-738e75872cb9



--- Tuning LightGBM with Optuna ---


[I 2026-06-04 19:41:42,955] Trial 0 finished with value: 0.06685945595127916 and parameters: {'learning_rate': 0.030032488873714507, 'max_depth': 7, 'num_leaves': 38, 'feature_fraction': 0.5725976542197253, 'bagging_fraction': 0.8432709151682047, 'bagging_freq': 4, 'min_child_samples': 31}. Best is trial 0 with value: 0.06685945595127916.
[I 2026-06-04 19:41:55,219] Trial 1 finished with value: 0.06717852347629343 and parameters: {'learning_rate': 0.01567335884885496, 'max_depth': 9, 'num_leaves': 57, 'feature_fraction': 0.694026337360995, 'bagging_fraction': 0.8075082898133097, 'bagging_freq': 5, 'min_child_samples': 25}. Best is trial 0 with value: 0.06685945595127916.
[I 2026-06-04 19:42:03,240] Trial 2 finished with value: 0.06690844568073595 and parameters: {'learning_rate': 0.015215009118661499, 'max_depth': 6, 'num_leaves': 34, 'feature_fraction': 0.5693539011656026, 'bagging_fraction': 0.7055869961782701, 'bagging_freq': 4, 'min_child_samples': 29}. Best is trial 0 with value: 

Best LightGBM Params: {'learning_rate': 0.026705362010972525, 'max_depth': 9, 'num_leaves': 31, 'feature_fraction': 0.8744492781237672, 'bagging_fraction': 0.7882193345226322, 'bagging_freq': 5, 'min_child_samples': 37}

--- Tuning CatBoost with Optuna ---


[I 2026-06-04 20:33:36,075] Trial 0 finished with value: 0.06360136838314964 and parameters: {'learning_rate': 0.02933450580177135, 'depth': 7, 'l2_leaf_reg': 9.930168991939329, 'random_strength': 1.5629008995517082, 'bagging_temperature': 0.5453356256225692}. Best is trial 0 with value: 0.06360136838314964.
[I 2026-06-04 20:35:11,947] Trial 1 finished with value: 0.06669614961328105 and parameters: {'learning_rate': 0.05017094802669502, 'depth': 9, 'l2_leaf_reg': 1.2116298074501564, 'random_strength': 0.7250720819457905, 'bagging_temperature': 0.924566726537984}. Best is trial 0 with value: 0.06360136838314964.
[I 2026-06-04 20:36:32,251] Trial 2 finished with value: 0.06453426504083085 and parameters: {'learning_rate': 0.02376484069993311, 'depth': 8, 'l2_leaf_reg': 1.7369565213718303, 'random_strength': 1.3009379388168054, 'bagging_temperature': 0.649065001777873}. Best is trial 0 with value: 0.06360136838314964.
[I 2026-06-04 20:37:17,099] Trial 3 finished with value: 0.06137785928

Best CatBoost Params: {'learning_rate': 0.044118509188828876, 'depth': 5, 'l2_leaf_reg': 5.244014741947192, 'random_strength': 0.6633702913182061, 'bagging_temperature': 0.23133479121719513}

--- Training Final CatBoost Model ---
0:	learn: 0.1364045	test: 0.1404205	best: 0.1404205 (0)	total: 32.8ms	remaining: 10m 55s
250:	learn: 0.0369873	test: 0.0651630	best: 0.0627631 (93)	total: 7.53s	remaining: 9m 52s
500:	learn: 0.0351416	test: 0.0671481	best: 0.0627631 (93)	total: 15.1s	remaining: 9m 48s
750:	learn: 0.0341406	test: 0.0689113	best: 0.0627631 (93)	total: 22.7s	remaining: 9m 40s
1000:	learn: 0.0335081	test: 0.0699632	best: 0.0627631 (93)	total: 30.3s	remaining: 9m 34s
1250:	learn: 0.0331471	test: 0.0706910	best: 0.0627631 (93)	total: 37.8s	remaining: 9m 26s
1500:	learn: 0.0327991	test: 0.0707702	best: 0.0627631 (93)	total: 45.4s	remaining: 9m 19s
1750:	learn: 0.0324501	test: 0.0711705	best: 0.0627631 (93)	total: 53s	remaining: 9m 11s
2000:	learn: 0.0322417	test: 0.0714698	best: 0.06